# k-ADAPT-VQE auto-ajusté + pools d'opérateurs mélangés

Suite de `VQE_benchmark.ipynb`. Objectif : **laisser l'algorithme trouver tout seul la meilleure configuration de k-ADAPT**, en testant d'autres ensembles d'opérateurs (pools) et des mélanges de pools, puis **vérifier que cette configuration se généralise** à des molécules qu'il n'a jamais vues (proxy de la molécule finale, « un peu plus complexe »).

**Principe**
1. Un **k-ADAPT généralisé** dont tout le comportement est piloté par quelques hyperparamètres :
   - `pool` : l'ensemble d'opérateurs (voir tableau) ;
   - `k_max` et `alpha` : **k dynamique** — à chaque itération on ajoute tous les opérateurs dont le score ≥ `alpha` × (meilleur score), au plus `k_max` ;
   - `beta` : **sélection sensible au coût** — score = |gradient| / (CNOT de l'opérateur)^`beta`. Avec un pool mélangé, ça permet de préférer l'opérateur bon marché quand deux opérateurs ont un gradient proche ;
   - `tetris` : n'ajouter ensemble que des opérateurs agissant sur des qubits **disjoints** (ils s'exécutent en parallèle → profondeur réduite) ;
   - `eps`, `plateau` : critères d'arrêt (norme du gradient / gain d'énergie minimal par itération) ;
   - `escape` : nombre max de **sorties de vallée** — quand tous les gradients sont ~0 alors que l'énergie n'est pas convergée, on teste quelques opérateurs en optimisant réellement leur angle (au lieu de s'arrêter) ;
   - `prune_tol` : **élagage** final **protégé** — on supprime les opérateurs dont |θ| < tol, mais seulement si l'énergie remonte de moins de `PRUNE_MAX_LOSS` (sinon on en supprime moins).
2. Un **auto-tuner** (Optuna, échantillonneur bayésien TPE) qui cherche la configuration minimisant un **score** : moins de CNOT et de circuits QPU, **sous contrainte** d'erreur < `SAFETY` × précision chimique sur **toutes** les molécules d'entraînement.
3. Une **validation** sur des molécules non vues (LiH étiré = forte corrélation, H₂O = 12 qubits / 8 électrons).
4. Application à la **molécule cible**.

**Pools disponibles**

| Pool | Contenu | CNOT / opérateur | Remarque |
|---|---|---|---|
| `QEB` | excitations qubit simples + doubles (Givens) | 2 / 14 | référence (meilleur du benchmark) |
| `QEB+GS` | doubles + **simples généralisées** (toutes paires d'orbitales de même spin) | 2 / 14 | plus expressif, peut sortir des « vallées » de gradient |
| `Pauli` | **qubit-ADAPT** : chaînes de Pauli (8 par double, 2 par simple) | 2 / 6 | opérateurs ~2× moins chers mais plus nombreux ; brise la symétrie → pénalité |
| `QEB+Pauli` | **mélange** des deux | 2–14 | avec `beta>0`, choisit le moins cher à gradient égal ; précis mais ~20× plus lent (pool 9× plus grand) |
| `QEB+Pauli1` | QEB + **une seule** chaîne de Pauli par excitation (`XXXY`, `XY`) | 2–14 | v5 : pool 2× QEB seulement → garde l'opérateur 6 CNOT sans l'explosion du temps |
| `QEB+GS+Pauli1` | QEB+GS + une chaîne de Pauli par excitation | 2–14 | v5 : mélange des deux meilleurs pools + Pauli léger |
| `Paired+S` | doubles **appariées** (paire αβ → paire αβ) + simples QEB | 2 / 14 | pool très petit (screening rapide) |
| `Paired+Pauli` | doubles appariées QEB + chaînes de Pauli | 2–14 | mélange |

> ⚠️ Les chaînes de Pauli seules ne conservent pas le nombre d'électrons ni Sz. Pour ces pools on minimise $\langle H\rangle + \mu\langle (N-N_e)^2 + (S_z - S_z^{cible})^2\rangle$ et on **vérifie** la fuite de symétrie à la fin (`fuite_N`, `fuite_Sz`). Sinon, pour un cation comme H₃⁺, l'algorithme pourrait « tricher » en ajoutant des électrons et trouver une énergie **sous** la FCI.

## Nouveautés v5 (d'après run_2)

| Constat run_2 | Correction v5 |
|---|---|
| BeH₂ étiré : 3 configs sur 4 s'arrêtent à 2,8 mHa (~200 CNOT) ; NH₃ et BeH₂ restent bloqués vers 0,4–1 mHa | **2 causes d'arrêt prématuré traitées** : (1) un opérateur choisi « bon marché » (β>0) qui ne fait rien gagner n'arrête plus l'algorithme — on retente avec le plus grand gradient, et l'ajout inutile est annulé ; (2) **sortie de vallée** (`escape`) quand les gradients sont ~0 mais que l'énergie peut encore baisser |
| Élagage à 0,02 : −CNOT à l'entraînement mais NH₃ passe de 0,95 à 2,05 mHa | **Élagage protégé** : perte d'énergie max `PRUNE_MAX_LOSS` = 0,05 mHa |
| NH₃ atteint `MAX_OPS = 60` (60 paramètres) | `MAX_OPS = 150`, colonne `arret` (« max_ops » = coupé), avertissement, et relance automatique avec une limite doublée pour la molécule cible |
| Le tuner exploite la marge (`SAFETY` 0,5 → configs à 0,52 mHa sur BeH₂) | **Précision d'abord** : `SAFETY = 0.25` (0,4 mHa), puis seulement minimisation des portes |
| H₃⁺ toujours à 0 mHa (n'aide pas à départager) ; échec sur BeH₂ étiré | Entraînement : LiH, BeH₂, **BeH₂ étiré**, H₂O ; validation : LiH étiré, NH₃, **N₂** (triple liaison) |
| QEB+Pauli : −15 % de CNOT mais ~20× plus lent | Nouveaux pools mélangés plus légers `QEB+Pauli1`, `QEB+GS+Pauli1` : le tuner les départage |
| VQE classique COBYLA bloqué à Hartree-Fock sur BeH₂ (76 évaluations en 5 min) | Classique = UCCSD + **gradients exacts + BFGS** (recette PennyLane standard), budget 10 min, temps pour atteindre la précision chimique |

## 0. Installation et imports

In [ ]:
import sys, subprocess, importlib.util
for pkg in ["pennylane", "optuna", "pandas", "matplotlib"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

# (Optionnel, Colab) sauvegarde sur Drive :
# from google.colab import drive; drive.mount("/content/drive")
# RESULTS_DIR_OVERRIDE = "/content/drive/MyDrive/hackathon_vqe"

In [ ]:
import os, time, json, warnings
from datetime import datetime
from itertools import combinations
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pennylane as qml
from pennylane import numpy as pnp
from scipy.optimize import minimize, minimize_scalar
from scipy.sparse.linalg import eigsh
import optuna

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
print("PennyLane", qml.__version__, "| Optuna", optuna.__version__)

## 1. Configuration

Les **poids du score** (`W_CNOT`, `W_HW`, `W_DEPTH`) traduisent ce qui compte pour le jury : adaptez-les dès que vous connaissez la grille de notation. Le score d'une configuration vaut, en moyenne sur les molécules d'entraînement :

$$\text{score} = W_{CNOT}\log\frac{\text{CNOT}}{\text{CNOT}_{ref}} + W_{TIME}\log\frac{\text{travail}}{\text{travail}_{ref}} +W_{HW}\log\frac{\text{circuits QPU}}{\text{circuits}_{ref}} + W_{DEPTH}\log\frac{\text{profondeur}}{\text{prof}_{ref}}$$

où *ref* = k-ADAPT (QEB, k=2) du benchmark. Le *travail* est le nombre de portes simulées (≈ temps statevector, mais **déterministe** : le chronomètre de Colab varie de ±30 % d'un essai à l'autre, ce qui faussait le score de run_1 — la référence y obtenait −0,19 au lieu de 0). **Score < 0 ⇒ meilleur que la référence.** Une molécule qui rate `SAFETY` × 1,6 mHa (ou qui fuit en symétrie) coûte +10 (+ l'écart en log).

In [ ]:
RUN_TAG = "autotune-v5"
BASIS = "sto-3g"
QCHEM_METHOD = "dhf"            # "dhf" partout ; "pyscf" plus rapide sur Linux/Colab (pip install pyscf)
DEVICE_NAME = "lightning.qubit"
CHEM_ACC = 1.6e-3               # Ha
RESULTS_DIR = globals().get("RESULTS_DIR_OVERRIDE", "results")
SEED = 0

# --- Molécules ---
# run_1 : sur H3+/LiH/BeH2 les 9 meilleures configs donnaient le MÊME circuit -> trop faciles pour départager.
# On entraîne sur des cas plus durs (H2O : 8 e-, 12 qubits) et on valide sur des cas encore plus proches de la
# "molécule un peu plus complexe" du jour J (liaisons étirées = forte corrélation, NH3 = 14 qubits).
# H3+ reste à l'entraînement : il est quasi gratuit et vérifie le cas cation (pas de fuite d'électrons).
# v5 : H3+ retiré (0 mHa pour toutes les configs), BeH2_étiré passe à l'entraînement (il faisait échouer 3 configs
# sur 4 en validation), N2 (triple liaison, très corrélée) rejoint la validation. Ordre = du moins cher au plus cher
# (arrêt anticipé dès qu'une molécule échoue).
TRAIN_MOLECULES = ["LiH", "BeH2", "BeH2_étiré", "H2O"]   # servent à ajuster les hyperparamètres
VALID_MOLECULES = ["LiH_étiré", "N2", "NH3"]             # jamais vues pendant l'ajustement : test de généralisation
TARGET_MOLECULE = None                       # nom de la molécule du jour J (à ajouter dans MOLECULES)

# --- Auto-tuner ---
N_TRIALS = 40                   # nb de configurations DIFFÉRENTES testées (~40-60 min sur Colab)
TUNER_TIMEOUT_MIN = 75          # arrêt de sécurité
# Priorité du jury : 1) précision chimique (contrainte dure via SAFETY)  2) nombre de portes  3) temps.
# Simulation statevector => le "temps" est le temps d'exécution réel (W_TIME) ; W_HW (circuits QPU) mis à 0.
W_CNOT, W_TIME, W_HW, W_DEPTH = 1.0, 0.2, 0.0, 0.0
SAFETY = 0.25                   # v5 : erreur < 0.25 x 1.6 = 0.4 mHa à l'entraînement (précision d'abord, portes ensuite)

# --- Comparaison avec un VQE classique (§8) ---
COMPARE_MOLECULES = ["H3+", "LiH", "BeH2"]
# v5 : COBYLA (sans gradient) ne bougeait pas de Hartree-Fock sur BeH2 en 5 min (204 paramètres) -> référence peu
# honnête. "BFGS" = UCCSD fermionique + gradients exacts (adjoint) : la recette PennyLane standard.
CLASSIC_OPTIMIZER = "BFGS"      # ou "COBYLA"
CLASSIC_MAXITER = 3000
CLASSIC_TIME_LIMIT_MIN = 10     # budget max par molécule : au-delà on arrête proprement (sans casser le run),
                                # on garde le meilleur point trouvé et le gain affiché est une borne basse (≥)
RUN_CLASSIC = True              # False pour sauter complètement la comparaison
PENALTY_MU = 1.0                # poids de la pénalité de symétrie (pools Pauli)
MAX_OPS = 150                   # taille max de l'ansatz (v5 : 60 était atteint sur NH3 -> calcul coupé)
GAIN_MIN = 1e-9                 # gain d'énergie (Ha) en dessous duquel un ajout est jugé inutile
PRUNE_MAX_LOSS = 5e-5           # élagage protégé : perte d'énergie max acceptée (Ha) = 0,05 mHa
ESCAPE_CANDIDATES = 8           # sortie de vallée : nb d'opérateurs testés (optimisation 1D de leur angle)
ESCAPE_MIN_GAIN = 1e-6          # gain min (Ha) pour accepter une sortie de vallée

# Étape "complet" du pipeline gelé -> non gelé (§8) : la corrélation du cœur est faible (~0,2-0,3 mHa),
# ses gradients sont < 1e-3, donc avec l'eps de l'entraînement aucune excitation de cœur n'était ajoutée.
# On utilise des critères plus fins pour cette seule étape (elle part d'un état déjà quasi convergé).
FULL_STAGE = dict(eps=1e-4, plateau=0.0)

# Espace v3, resserré d'après les runs 1 et 2 (§6.2) :
#  - pools : QEB et QEB+GS toujours < 0 et stables ; Pauli / Paired+Pauli presque toujours > 0 (lents) -> retirés,
#    QEB+Pauli gardé car il donne les circuits les plus courts (utile si on baisse W_TIME)
#  - prune_tol=3e-2 : 0 % de réussite -> retiré ; on explore autour de 1e-2
#  - eps=1e-4 et plateau=1e-5 dégradent le score -> retirés ; alpha élevé (0,5-0,75) est meilleur -> on ajoute 0,9
# v5 : + pools Pauli légers, + eps=2e-4 (BeH2 restait bloqué vers 0,4 mHa avec 1e-3), + escape (sorties de vallée).
SEARCH_SPACE = dict(
    pool=["QEB", "QEB+GS", "QEB+Pauli", "QEB+Pauli1", "QEB+GS+Pauli1"],
    k_max=[2, 3, 4, 5],
    alpha=[0.0, 0.5, 0.75, 0.9],
    beta=[0.0, 0.5, 1.0],
    tetris=[False, True],
    eps=[1e-3, 5e-4, 2e-4],
    plateau=[0.0, 1e-6],
    prune_tol=[0.0, 5e-3, 1e-2, 2e-2],
    escape=[0, 2, 5],
)
BASELINE_CFG = dict(pool="QEB", k_max=2, alpha=0.0, beta=0.0, tetris=False,
                    eps=1e-3, plateau=0.0, prune_tol=0.0, escape=0)   # = k-ADAPT (QEB, k=2) du benchmark
# Configurations testées en premier (doivent appartenir à SEARCH_SPACE) : référence + meilleures des runs précédents
SEED_CONFIGS = [
    BASELINE_CFG,
    dict(pool="QEB", k_max=5, alpha=0.5, beta=0.5, tetris=False, eps=5e-4, plateau=0.0, prune_tol=2e-2, escape=0),  # meilleur run 2
    dict(pool="QEB", k_max=5, alpha=0.5, beta=0.5, tetris=False, eps=5e-4, plateau=0.0, prune_tol=2e-2, escape=2),  # idem + vallées
    dict(pool="QEB+GS", k_max=5, alpha=0.5, beta=0.5, tetris=False, eps=1e-3, plateau=1e-6, prune_tol=1e-2, escape=2),  # meilleur run 1
    dict(pool="QEB", k_max=5, alpha=0.5, beta=0.5, tetris=False, eps=2e-4, plateau=0.0, prune_tol=1e-2, escape=2),
    dict(pool="QEB+Pauli1", k_max=4, alpha=0.5, beta=0.5, tetris=False, eps=5e-4, plateau=0.0, prune_tol=1e-2, escape=2),
    dict(pool="QEB+GS+Pauli1", k_max=4, alpha=0.5, beta=0.5, tetris=False, eps=5e-4, plateau=0.0, prune_tol=1e-2, escape=2),
    dict(pool="QEB+Pauli", k_max=4, alpha=0.5, beta=0.0, tetris=True, eps=5e-4, plateau=1e-6, prune_tol=1e-2, escape=0),  # le - de CNOT en run 2
]

# --- Géométries (Angström) ---
MOLECULES = {
    "H3+": dict(symbols=["H", "H", "H"],
                coords=[[0, 0, 0], [0.874, 0, 0], [0.437, 0.7569, 0]],
                charge=1, mult=1, active_electrons=None, active_orbitals=None),
    "LiH": dict(symbols=["Li", "H"], coords=[[0, 0, 0], [0, 0, 1.5949]],
                charge=0, mult=1, active_electrons=2, active_orbitals=5),
    "BeH2": dict(symbols=["Be", "H", "H"], coords=[[0, 0, 0], [0, 0, 1.326], [0, 0, -1.326]],
                 charge=0, mult=1, active_electrons=4, active_orbitals=6),
    # --- validation ---
    "LiH_étiré": dict(symbols=["Li", "H"], coords=[[0, 0, 0], [0, 0, 2.8]],      # liaison étirée : forte corrélation
                      charge=0, mult=1, active_electrons=2, active_orbitals=5),
    "H2O": dict(symbols=["O", "H", "H"], coords=[[0, 0, 0], [0.7572, 0.5865, 0], [-0.7572, 0.5865, 0]],
                charge=0, mult=1, active_electrons=8, active_orbitals=6),    # 12 qubits, 8 e-
    # Autres candidats plus durs (plus lents) :
    "BeH2_étiré": dict(symbols=["Be", "H", "H"], coords=[[0, 0, 0], [0, 0, 2.0], [0, 0, -2.0]],
                       charge=0, mult=1, active_electrons=4, active_orbitals=6),   # 12 qubits, forte corrélation
    "NH3": dict(symbols=["N", "H", "H", "H"],
                coords=[[0, 0, 0.1173], [0, 0.9377, -0.2737], [0.8121, -0.4689, -0.2737], [-0.8121, -0.4689, -0.2737]],
                charge=0, mult=1, active_electrons=8, active_orbitals=7),          # cœur N 1s gelé : 14 qubits
    "N2": dict(symbols=["N", "N"], coords=[[0, 0, 0], [0, 0, 1.098]],
               charge=0, mult=1, active_electrons=6, active_orbitals=6),           # triple liaison (2p) : 12 qubits, 6 e-
}

## 2. Problème : hamiltonien, HF, FCI, opérateurs de symétrie

In [ ]:
ANGSTROM_TO_BOHR = 1.8897259886

def symmetry_penalty(nq, ne, two_sz):
    # (N - Ne)^2 + (Sz - Sz_cible)^2 écrit en Pauli Z (mapping Jordan-Wigner, spin-orbitales alternées a,b).
    c = nq / 2 - ne
    m = two_sz / 2
    s = np.array([-0.25 if i % 2 == 0 else 0.25 for i in range(nq)])       # Sz = sum_i s_i Z_i
    coeffs = [c * c + nq / 4 + np.sum(s ** 2) + m * m]
    ops = [qml.Identity(0)]
    for i in range(nq):
        coeffs.append(-c - 2 * m * s[i]); ops.append(qml.PauliZ(i))
    for i, j in combinations(range(nq), 2):
        coeffs.append(0.5 + 2 * s[i] * s[j]); ops.append(qml.PauliZ(i) @ qml.PauliZ(j))
    return qml.Hamiltonian(coeffs, ops)

_PROBLEMS = {}
def get_problem(name):
    if name in _PROBLEMS:
        return _PROBLEMS[name]
    cfg = MOLECULES[name]
    t0 = time.time()
    coords = np.array(cfg["coords"], dtype=float) * ANGSTROM_TO_BOHR
    mol = qml.qchem.Molecule(cfg["symbols"], coords, charge=cfg["charge"], mult=cfg["mult"], basis_name=BASIS)
    ne = cfg["active_electrons"] or mol.n_electrons
    H, nq = qml.qchem.molecular_hamiltonian(mol, active_electrons=cfg["active_electrons"],
                                            active_orbitals=cfg["active_orbitals"],
                                            method=QCHEM_METHOD, mapping="jordan_wigner")
    wires = list(range(nq))
    hf = np.array(qml.qchem.hf_state(ne, nq))
    Hs = H.sparse_matrix(wire_order=wires).tocsr()
    idx = np.arange(2 ** nq)
    bits = (idx[:, None] >> (nq - 1 - np.arange(nq))[None, :]) & 1
    n_occ = bits.sum(1)
    two_sz = bits[:, 0::2].sum(1) - bits[:, 1::2].sum(1)
    target_2sz = cfg["mult"] - 1
    sel = np.where((n_occ == ne) & (two_sz == target_2sz))[0]
    Hsub = Hs[sel][:, sel]
    e_fci = np.linalg.eigvalsh(Hsub.toarray()).min() if len(sel) <= 3000 else eigsh(Hsub, k=1, which="SA")[0][0]
    e_hf = Hs[int("".join(map(str, hf)), 2), int("".join(map(str, hf)), 2)].real
    singles, doubles = qml.qchem.excitations(ne, nq)
    prob = dict(name=name, H=H, Hs=Hs, nq=nq, ne=ne, wires=wires, hf_state=hf,
                E_fci=float(e_fci), E_hf=float(e_hf), singles=singles, doubles=doubles,
                n_occ=n_occ, two_sz=two_sz, target_2sz=target_2sz,
                Pen=symmetry_penalty(nq, ne, target_2sz),
                n_core=(mol.n_electrons - ne) // 2, t_build=time.time() - t0)
    print(f"{name:10s} | {nq:2d} qubits | {ne} e- | {len(singles)} S, {len(doubles)} D | "
          f"E_FCI = {e_fci:.6f} Ha | corrélation = {1e3*(e_hf-e_fci):.2f} mHa | {time.time()-t0:.1f}s")
    _PROBLEMS[name] = prob
    return prob

def full_name(name):
    # Variante SANS gel de cœur (toutes les orbitales actives) = équivalent de "sans FreezeCoreTransformer".
    if MOLECULES[name]["active_electrons"] is None:
        return name
    fn = name + "_complet"
    if fn not in MOLECULES:
        MOLECULES[fn] = dict(MOLECULES[name], active_electrons=None, active_orbitals=None)
    return fn

## 3. Opérateurs et pools

In [ ]:
# Un opérateur = (type, wires). Types : S, D (qubit/Givens), FS, FD (fermionique), P (chaîne de Pauli : wires = (mot, qubits))
def apply_op(op, theta):
    kind, w = op
    if kind == "S":    qml.SingleExcitation(theta, wires=list(w))
    elif kind == "D":  qml.DoubleExcitation(theta, wires=list(w))
    elif kind == "FS": qml.FermionicSingleExcitation(theta, wires=list(w))
    elif kind == "FD": qml.FermionicDoubleExcitation(theta, wires1=list(w[0]), wires2=list(w[1]))
    elif kind == "P":  qml.PauliRot(theta, w[0], wires=list(w[1]))
    else: raise ValueError(kind)

def apply_ansatz(ops, params):
    for op, t in zip(ops, params):
        apply_op(op, t)

def op_wires(op):
    kind, w = op
    if kind == "FD": return set(w[0]) | set(w[1])
    if kind == "P":  return set(w[1])
    return set(w)

# ---- briques de pools ----
def qeb_doubles(p):  return [("D", tuple(d)) for d in p["doubles"]]
def qeb_singles(p):  return [("S", tuple(s)) for s in p["singles"]]
def gen_singles(p):  # toutes les paires de même spin (pas seulement occupé -> virtuel)
    return [("S", (i, j)) for i, j in combinations(range(p["nq"]), 2) if i % 2 == j % 2]
def paired_doubles(p):  # paire (i_a, i_b) -> (a_a, a_b)
    return [("D", tuple(d)) for d in p["doubles"] if d[0] // 2 == d[1] // 2 and d[2] // 2 == d[3] // 2]
ODD_Y_4 = ["XXXY", "XXYX", "XYXX", "YXXX", "YYYX", "YYXY", "YXYY", "XYYY"]
def pauli_singles(p): return [("P", (wd, tuple(s))) for s in p["singles"] for wd in ("XY", "YX")]
def pauli_doubles(p): return [("P", (wd, tuple(d))) for d in p["doubles"] for wd in ODD_Y_4]
def pauli1_singles(p): return [("P", ("XY", tuple(s))) for s in p["singles"]]
def pauli1_doubles(p): return [("P", ("XXXY", tuple(d))) for d in p["doubles"]]
def fermionic(p):
    s_w, d_w = qml.qchem.excitations_to_wires(p["singles"], p["doubles"])
    return [("FD", (tuple(a), tuple(b))) for a, b in d_w] + [("FS", tuple(w)) for w in s_w]

POOLS = {
    "QEB":          lambda p: qeb_doubles(p) + qeb_singles(p),
    "QEB+GS":       lambda p: qeb_doubles(p) + gen_singles(p),
    "Pauli":        lambda p: pauli_doubles(p) + pauli_singles(p),
    "QEB+Pauli":    lambda p: qeb_doubles(p) + qeb_singles(p) + pauli_doubles(p) + pauli_singles(p),
    "QEB+Pauli1":   lambda p: qeb_doubles(p) + qeb_singles(p) + pauli1_doubles(p) + pauli1_singles(p),
    "QEB+GS+Pauli1": lambda p: qeb_doubles(p) + gen_singles(p) + pauli1_doubles(p) + pauli1_singles(p),
    "Paired+S":     lambda p: paired_doubles(p) + qeb_singles(p),
    "Paired+Pauli": lambda p: paired_doubles(p) + pauli_doubles(p) + pauli_singles(p),
    "Fermionic":    fermionic,
}
def pool_breaks_symmetry(pool):
    return any(k == "P" for k, _ in pool)

In [ ]:
def _decompose(op):
    if len(op.wires) == 1 or op.name in ("CNOT", "CZ"):
        return [op]
    try:
        dec = op.decomposition()
    except Exception:
        return [op]
    out = []
    for o in dec:
        out += _decompose(o)
    return out

def circuit_resources(ops):
    with qml.queuing.AnnotatedQueue() as q:
        apply_ansatz(ops, np.full(len(ops), 0.1))
    tape = qml.tape.QuantumScript.from_queue(q)
    gates = [g for op in tape.operations for g in _decompose(op) if g.name not in ("GlobalPhase", "Identity")]
    layer = {}
    for g in gates:
        d = max(layer.get(w, 0) for w in g.wires) + 1
        for w in g.wires:
            layer[w] = d
    return dict(n_cnot=sum(len(g.wires) >= 2 for g in gates), n_portes=len(gates),
                profondeur=max(layer.values()) if layer else 0, n_params=len(ops),
                n_cnot_simplifie=sum(len(g.wires) >= 2 for g in cancel_cnots(gates)))

def cancel_cnots(gates):
    # Simplification "peephole" : deux CNOT identiques consécutifs sur les mêmes qubits (aucune porte entre eux
    # sur ces qubits) s'annulent. Répété jusqu'à stabilité. Utile entre deux excitations voisines de l'ansatz.
    gates = list(gates)
    while True:
        out, last, removed = [], {}, False
        for g in gates:
            if g.name == "CNOT":
                a, b = list(g.wires)
                i = last.get(a)
                if i is not None and i == last.get(b) and out[i].name == "CNOT" and list(out[i].wires) == [a, b]:
                    out[i] = None; removed = True
                    for w in (a, b):            # recule le pointeur vers la porte précédente sur ce qubit
                        k = i - 1
                        while k >= 0 and (out[k] is None or w not in out[k].wires):
                            k -= 1
                        last[w] = k if k >= 0 else None
                    continue
            out.append(g)
            for w in g.wires:
                last[w] = len(out) - 1
        gates = [g for g in out if g is not None]
        if not removed:
            return gates

_COST = {}
def op_cost(op):
    if op not in _COST:
        _COST[op] = max(circuit_resources([op])["n_cnot"], 1)
    return _COST[op]

# Coût de chaque type d'opérateur (vérification)
_p = get_problem("H3+")
for name, ops in [("QEB double", qeb_doubles(_p)[:1]), ("QEB simple", qeb_singles(_p)[:1]),
                  ("Pauli 4 qubits", pauli_doubles(_p)[:1]), ("Pauli 2 qubits", pauli_singles(_p)[:1]),
                  ("Fermionique double", fermionic(_p)[:1])]:
    print(f"{name:20s}: {op_cost(ops[0])} CNOT")

## 4. Simulateur, optimiseur et k-ADAPT généralisé

In [ ]:
class Estimator:
    def __init__(self, prob, penalty=False):
        self.prob = prob
        self.H = prob["H"] + PENALTY_MU * prob["Pen"] if penalty else prob["H"]
        try:
            self.dev = qml.device(DEVICE_NAME, wires=prob["nq"])
        except Exception:
            self.dev = qml.device("default.qubit", wires=prob["nq"])
        self._key, self._qn = None, None
        self.n_energy = self.n_grad = self.n_hw = 0
        # Coût de simulation DÉTERMINISTE (proportionnel au temps statevector, sans le bruit de mesure du chrono) :
        # nb de portes appliquées, en comptant ~3 passes pour un gradient adjoint (aller + retour).
        self.travail = 0

    def _qnode(self, ops):
        key = tuple(ops)
        if key != self._key:
            prob, H = self.prob, self.H
            @qml.qnode(self.dev, diff_method="adjoint")
            def circuit(params):
                qml.BasisState(prob["hf_state"], wires=prob["wires"])
                apply_ansatz(ops, params)
                return qml.expval(H)
            self._key, self._qn = key, circuit
        return self._qn

    def energy(self, ops, x):
        self.n_energy += 1; self.n_hw += 1; self.travail += len(ops)
        if len(x) == 0:
            return self.prob["E_hf"]
        return float(self._qnode(ops)(pnp.array(x, requires_grad=True)))

    def grad(self, ops, x, n_hw_params=None):
        self.n_grad += 1; self.travail += 3 * len(ops)
        self.n_hw += 2 * (len(x) if n_hw_params is None else n_hw_params)
        return np.asarray(qml.grad(self._qnode(ops))(pnp.array(x, requires_grad=True)), dtype=float)

    def final_stats(self, ops, x):
        # Énergie "pure" <H> (sans pénalité) + fuite de symétrie, calculées sur le vecteur d'état exact.
        prob = self.prob
        @qml.qnode(self.dev)
        def st(params):
            qml.BasisState(prob["hf_state"], wires=prob["wires"])
            apply_ansatz(ops, params)
            return qml.state()
        psi = np.asarray(st(np.array(x, dtype=float)))
        pr = np.abs(psi) ** 2
        E = float(np.real(np.vdot(psi, prob["Hs"] @ psi)))
        return dict(E=E, fuite_N=abs(pr @ prob["n_occ"] - prob["ne"]),
                    fuite_Sz=abs(pr @ prob["two_sz"] - prob["target_2sz"]) / 2)


def optimize(est, ops, x0, free=None, maxiter=500):
    x0 = np.array(x0, dtype=float)
    if len(x0) == 0:
        return x0, est.energy(ops, x0)
    free = np.arange(len(x0)) if free is None else np.array(free)
    def full(y):
        x = x0.copy(); x[free] = y; return x
    res = minimize(lambda y: est.energy(ops, full(y)), x0[free],
                   jac=lambda y: est.grad(ops, full(y), n_hw_params=len(free))[free],
                   method="BFGS", options=dict(gtol=1e-6, maxiter=maxiter))
    return full(res.x), float(res.fun)

In [ ]:
def _lookahead(est, ops, x, E, pool, g):
    # Sortie de vallée : dans une "vallée de gradient", tous les gradients sont ~0 (au 1er ordre rien ne bouge) alors
    # que l'énergie peut encore baisser (2e ordre / couplage). On teste les ESCAPE_CANDIDATES opérateurs de plus grand
    # |g| en optimisant VRAIMENT leur angle (1D, autres angles fixés) et on garde celui qui fait le plus baisser E.
    best_c, best_gain, best_t = None, 0.0, 0.0
    for c in np.argsort(-np.abs(g))[:ESCAPE_CANDIDATES]:
        trial_ops = list(ops) + [pool[int(c)]]
        r = minimize_scalar(lambda t: est.energy(trial_ops, np.append(x, t)), bounds=(-1.0, 1.0),
                            method="bounded", options=dict(xatol=1e-4, maxiter=30))
        if E - r.fun > best_gain:
            best_c, best_gain, best_t = int(c), E - r.fun, float(r.x)
    return best_c, best_gain, best_t


def run_auto_adapt(est, prob, cfg, pool, init=None, max_ops=None):
    max_ops = max_ops or MAX_OPS
    costs = np.array([op_cost(o) for o in pool], dtype=float)
    ops, x, E = [], np.zeros(0), prob["E_hf"]
    if init is not None:                       # warm start (ex. ansatz trouvé dans l'espace gelé)
        ops, x = list(init[0]), np.array(init[1], dtype=float)
        x, E = optimize(est, ops, x)
    t_start = time.time()
    hist = [dict(n_ops=len(ops), E=E, n_hw=est.n_hw, t=0.0)]
    escapes, stop = 0, "max_ops"

    def _try(idx, theta0=None):
        new_ops = list(ops) + [pool[c] for c in idx]
        x0 = np.append(x, np.zeros(len(idx)) if theta0 is None else theta0)
        xn, En = optimize(est, new_ops, x0)
        return new_ops, xn, En

    def _useless(gain):
        return gain < GAIN_MIN or (cfg["plateau"] > 0 and gain < cfg["plateau"])

    while len(ops) < max_ops:
        # 1) gradients de tout le pool (une passe adjoint ; sur QPU : 2 circuits / opérateur)
        g = est.grad(list(ops) + pool, np.concatenate([x, np.zeros(len(pool))]), n_hw_params=len(pool))[len(x):]
        chosen = []
        if np.linalg.norm(g) >= cfg["eps"]:
            # 2) sélection : score sensible au coût, k dynamique, option TETRIS (qubits disjoints)
            score = np.abs(g) / costs ** cfg["beta"]
            order = np.argsort(-score)
            smax = score[order[0]]
            used = set()
            for c in order:
                if len(chosen) >= min(cfg["k_max"], max_ops - len(ops)) or score[c] < cfg["alpha"] * smax \
                        or abs(g[c]) < 1e-10:
                    break
                w = op_wires(pool[c])
                if cfg["tetris"] and (w & used):
                    continue
                chosen.append(int(c)); used |= w
        if chosen:
            # 3) réoptimisation complète (warm start)
            new_ops, xn, En = _try(chosen)
            gain = E - En
            if _useless(gain):
                # Cause 1 d'arrêt prématuré : avec beta>0 on a pu choisir un opérateur bon marché mais inutile.
                # On retente avec l'opérateur de plus grand |gradient| (= sélection ADAPT pure) avant d'abandonner.
                c0 = int(np.argmax(np.abs(g)))
                if c0 not in chosen:
                    o2, x2, E2 = _try([c0])
                    if E - E2 > gain:
                        new_ops, xn, En, gain = o2, x2, E2, E - E2
            if not _useless(gain):
                ops, x, E = new_ops, xn, En
                hist.append(dict(n_ops=len(ops), E=E, n_hw=est.n_hw, t=time.time() - t_start))
                continue
            # l'ajout ne fait rien gagner : on NE le garde PAS (pas de CNOT pour rien)
        # Cause 2 : vallée de gradient (gradients ~0 ou ajouts inutiles alors que E peut encore baisser)
        if escapes < cfg.get("escape", 0) and len(ops) < max_ops:
            c, g1, t1 = _lookahead(est, ops, x, E, pool, g)
            if c is not None and g1 > ESCAPE_MIN_GAIN:
                # angle initial = optimum 1D (partir de 0 avec un gradient ~0 bloquerait BFGS)
                ops, x, E = _try([c], theta0=[t1])
                escapes += 1
                hist.append(dict(n_ops=len(ops), E=E, n_hw=est.n_hw, t=time.time() - t_start, escape=True))
                continue
        stop = "gradient" if not chosen else "plateau"
        break
    # 4) élagage PROTÉGÉ : on retire les |θ| < prune_tol seulement si E remonte de moins de PRUNE_MAX_LOSS ;
    #    sinon on retente avec la moitié (les plus petits angles), etc.
    if cfg["prune_tol"] > 0 and len(x):
        cand = [int(i) for i in np.argsort(np.abs(x)) if abs(x[i]) < cfg["prune_tol"]]
        while cand and len(cand) < len(x):
            keep = np.ones(len(x), dtype=bool); keep[cand] = False
            o2 = [o for o, k in zip(ops, keep) if k]
            x2, E2 = optimize(est, o2, x[keep])
            if E2 - E <= PRUNE_MAX_LOSS:
                ops, x, E = o2, x2, E2
                hist.append(dict(n_ops=len(ops), E=E, n_hw=est.n_hw, t=time.time() - t_start, pruned=True))
                break
            cand = cand[:len(cand) // 2]
    hist[-1]["stop"] = stop
    return ops, x, hist, dict(arret=stop, n_escapes=escapes)


def run_config(mol_name, cfg, label=None, init=None, max_ops=None):
    prob = get_problem(mol_name)
    pool = POOLS[cfg["pool"]](prob)
    est = Estimator(prob, penalty=pool_breaks_symmetry(pool) or (init is not None and pool_breaks_symmetry(init[0])))
    t0 = time.time()
    ops, x, hist, info = run_auto_adapt(est, prob, cfg, pool, init=init, max_ops=max_ops)
    stats = est.final_stats(ops, x)
    res = circuit_resources(ops)
    err = abs(stats["E"] - prob["E_fci"])
    row = dict(run_tag=RUN_TAG, molecule=mol_name, label=label or cfg_label(cfg), **cfg,
               n_qubits=prob["nq"], taille_pool=len(pool), E=stats["E"], E_fci=prob["E_fci"],
               erreur_mHa=1e3 * err, precision_chimique=err < CHEM_ACC,
               fuite_N=stats["fuite_N"], fuite_Sz=stats["fuite_Sz"],
               valide=bool(stats["fuite_N"] < 1e-3 and stats["fuite_Sz"] < 1e-3),
               **res, n_circuits_hw=est.n_hw, travail_sim=est.travail, temps_s=time.time() - t0, **info)
    if info["arret"] == "max_ops":
        print(f"   ⚠️  {mol_name} : limite MAX_OPS={max_ops or MAX_OPS} atteinte -> calcul coupé, augmentez MAX_OPS")
    for h in hist:
        h["err_mHa"] = 1e3 * abs(h["E"] - prob["E_fci"])
    return row, dict(ops=ops, x=x, hist=hist)

def cfg_label(cfg):
    return (f"{cfg['pool']} k≤{cfg['k_max']} α={cfg['alpha']} β={cfg['beta']}"
            f"{' T' if cfg['tetris'] else ''} ε={cfg['eps']:g} pl={cfg['plateau']:g} pr={cfg['prune_tol']:g}"
            f"{' esc=' + str(cfg['escape']) if cfg.get('escape') else ''}")

## 5. Référence et premier test des pools

On lance la référence (k-ADAPT QEB k=2) puis chaque pool avec les mêmes autres réglages, pour voir l'effet du pool seul.

In [ ]:
for m in dict.fromkeys(TRAIN_MOLECULES + VALID_MOLECULES):
    get_problem(m)

refs, pool_rows, runs = {}, [], {}
for m in TRAIN_MOLECULES:
    row, run = run_config(m, BASELINE_CFG, label="Référence k-ADAPT QEB k=2")
    refs[m] = row
    pool_rows.append(row); runs[(m, "Référence")] = run

for pool_name in SEARCH_SPACE["pool"]:
    if pool_name == "QEB":
        continue
    cfg = dict(BASELINE_CFG, pool=pool_name, beta=1.0 if "+" in pool_name else 0.0)
    for m in TRAIN_MOLECULES:
        row, run = run_config(m, cfg, label=f"pool {pool_name}" + (" (β=1)" if cfg["beta"] else ""))
        pool_rows.append(row); runs[(m, row["label"])] = run

pool_df = pd.DataFrame(pool_rows)
display(pool_df.pivot_table(index="label", columns="molecule",
                            values=["erreur_mHa", "n_cnot", "n_circuits_hw"], sort=False).round(3))

In [ ]:
fig, axes = plt.subplots(1, len(TRAIN_MOLECULES), figsize=(6 * len(TRAIN_MOLECULES), 4.5), squeeze=False)
labels = list(dict.fromkeys(pool_df.label))
colors = {l: plt.get_cmap("tab10")(i) for i, l in enumerate(labels)}
for ax, m in zip(axes[0], TRAIN_MOLECULES):
    for l in labels:
        r = runs.get((m, "Référence" if l.startswith("Référence") else l))
        if r is None: continue
        h = r["hist"]
        ax.plot([e["n_ops"] for e in h], np.maximum([e["err_mHa"] for e in h], 1e-6), marker="o", ms=3,
                color=colors[l], label=l)
    ax.axhline(1e3 * CHEM_ACC, color="green", ls="--")
    ax.set(yscale="log", xlabel="Nombre d'opérateurs", ylabel="Erreur (mHa)", title=m)
    ax.grid(alpha=0.3, which="both")
axes[0][0].legend(fontsize=8)
plt.suptitle("Convergence par pool"); plt.tight_layout(); plt.show()

## 6. Auto-tuner (Optuna / TPE)

Chaque essai = une configuration, évaluée sur toutes les molécules d'entraînement (de la moins chère à la plus chère ; arrêt anticipé dès qu'une molécule échoue). Les configurations déjà testées sont mises en cache.

In [ ]:
_cache, trial_rows = {}, []
FAIL = 10.0

def evaluate_config(cfg):
    key = tuple(sorted(cfg.items()))
    if key in _cache:
        return _cache[key]
    total, rows = 0.0, []
    for i, m in enumerate(TRAIN_MOLECULES):
        row, _ = run_config(m, cfg)
        rows.append(row)
        ref = refs[m]
        lim = SAFETY * 1e3 * CHEM_ACC
        if not row["valide"] or row["erreur_mHa"] > lim:
            total += FAIL + np.log10(max(row["erreur_mHa"], lim) / lim)
            total += FAIL * (len(TRAIN_MOLECULES) - i - 1)           # molécules non évaluées
            break
        total += (W_CNOT * np.log(max(row["n_cnot"], 1) / max(ref["n_cnot"], 1))
                  + W_TIME * np.log(max(row["travail_sim"], 1) / max(ref["travail_sim"], 1))
                  + W_HW * np.log(row["n_circuits_hw"] / ref["n_circuits_hw"])
                  + W_DEPTH * np.log(max(row["profondeur"], 1) / max(ref["profondeur"], 1)))
    score = total / len(TRAIN_MOLECULES)
    _cache[key] = (score, rows)
    return score, rows

def objective(trial):
    cfg = {k: trial.suggest_categorical(k, v) for k, v in SEARCH_SPACE.items()}
    t0 = time.time()
    score, rows = evaluate_config(cfg)
    for r in rows:
        trial_rows.append(dict(r, trial=trial.number, score=score))
    errs = " ".join(f"{r['molecule']}:{r['erreur_mHa']:.3f}mHa/{r['n_cnot']}CNOT"
                    + ("⚠MAX" if r["arret"] == "max_ops" else "") for r in rows)
    print(f"#{trial.number:3d} score={score:+7.3f} | {cfg_label(cfg):60s} | {errs} | {time.time()-t0:.0f}s")
    return score

study = optuna.create_study(direction="minimize",
                            sampler=optuna.samplers.TPESampler(seed=SEED, n_startup_trials=10))
for c in SEED_CONFIGS:
    study.enqueue_trial(c)

def _stop_when_enough(study, trial):
    # TPE repropose souvent une config déjà testée (servie par le cache, instantanée) :
    # on compte les configurations DIFFÉRENTES, pas les essais.
    if len(_cache) >= N_TRIALS or len(study.trials) >= 5 * N_TRIALS:
        study.stop()

study.optimize(objective, n_trials=None, timeout=TUNER_TIMEOUT_MIN * 60, callbacks=[_stop_when_enough])
print(f"{len(_cache)} configurations différentes évaluées en {len(study.trials)} essais")

best_cfg = dict(study.best_params)
print("\nMeilleure configuration :", cfg_label(best_cfg), f"| score = {study.best_value:+.3f}")

trials_df = pd.DataFrame(trial_rows)
os.makedirs(RESULTS_DIR, exist_ok=True)
p = os.path.join(RESULTS_DIR, "autotune_trials.csv")
trials_df.assign(date=datetime.now().strftime("%Y-%m-%d %H:%M")).to_csv(
    p, mode="a", header=not os.path.exists(p), index=False)
with open(os.path.join(RESULTS_DIR, "best_config.json"), "w") as f:
    json.dump(dict(cfg=best_cfg, score=study.best_value, run_tag=RUN_TAG,
                   weights=dict(W_CNOT=W_CNOT, W_TIME=W_TIME, W_HW=W_HW, W_DEPTH=W_DEPTH, SAFETY=SAFETY)), f, indent=2)

### 6.1 Top 10 des configurations

In [ ]:
summ = (trials_df.groupby("trial")
        .agg(score=("score", "first"), label=("label", "first"),
             n_ok=("precision_chimique", "sum"), cnot_total=("n_cnot", "sum"),
             circuits_total=("n_circuits_hw", "sum"), err_max_mHa=("erreur_mHa", "max"),
             escapes=("n_escapes", "sum"), n_max_ops=("arret", lambda s: int((s == "max_ops").sum())))
        .sort_values("score"))
display(summ.drop_duplicates("label").head(10).round(4))

### 6.2 Progression de l'auto-tuner et effet de chaque hyperparamètre

In [ ]:
sc = summ.sort_index()["score"]
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.scatter(sc.index, sc.clip(upper=5), c=np.where(sc < FAIL / len(TRAIN_MOLECULES), "tab:blue", "tab:red"), s=20)
ax.plot(sc.index, sc.cummin().clip(upper=5), color="k", label="meilleur jusqu'ici")
ax.axhline(0, color="gray", ls="--", label="référence k-ADAPT k=2")
ax.set(xlabel="Essai", ylabel="Score (plus bas = mieux, écrêté à 5)", title="Auto-tuner")
ax.legend(); ax.grid(alpha=0.3); plt.show()

ok = trials_df.drop_duplicates("trial").set_index("trial")
ok = ok[ok.score < FAIL / len(TRAIN_MOLECULES)]          # essais qui passent la contrainte de précision
fig, axes = plt.subplots(2, 5, figsize=(22, 7))
for ax, k in zip(axes.flat, SEARCH_SPACE):
    vals = SEARCH_SPACE[k]
    data = [ok[ok[k] == v].score.values for v in vals]
    allv = trials_df.drop_duplicates("trial")
    rate = [100 * np.mean(allv[allv[k] == v].score < FAIL / len(TRAIN_MOLECULES)) if (allv[k] == v).any() else np.nan
            for v in vals]
    ax.boxplot([d if len(d) else [np.nan] for d in data])
    ax.set_xticks(range(1, len(vals) + 1), [str(v) for v in vals])
    for i, r in enumerate(rate):
        ax.text(i + 1, ax.get_ylim()[1], f"{r:.0f}% ok" if np.isfinite(r) else "-", ha="center", va="bottom", fontsize=8)
    ax.axhline(0, color="gray", ls="--"); ax.set_title(k, pad=14); ax.tick_params(axis="x", rotation=30)
plt.suptitle("Score des configurations réussies selon chaque hyperparamètre (+ % de réussite)")
plt.tight_layout(); plt.show()

### 6.3 Toutes les configurations : précision vs CNOT

In [ ]:
pool_colors = {p: plt.get_cmap("tab10")(i) for i, p in enumerate(SEARCH_SPACE["pool"])}
best_trial = study.best_trial.number
fig, axes = plt.subplots(1, len(TRAIN_MOLECULES), figsize=(6 * len(TRAIN_MOLECULES), 4.8), squeeze=False)
for ax, m in zip(axes[0], TRAIN_MOLECULES):
    d = trials_df[trials_df.molecule == m]
    for p_, g in d.groupby("pool"):
        ax.scatter(g.n_cnot, np.maximum(g.erreur_mHa, 1e-6), color=pool_colors[p_], label=p_, alpha=0.7,
                   marker="o", edgecolor=np.where(g.valide, "none", "red"))
    b = d[d.trial == best_trial]
    ax.scatter(b.n_cnot, np.maximum(b.erreur_mHa, 1e-6), marker="*", s=350, color="gold", edgecolor="k",
               label="meilleure config", zorder=5)
    ax.scatter(refs[m]["n_cnot"], max(refs[m]["erreur_mHa"], 1e-6), marker="s", s=120, color="k", label="référence")
    ax.axhline(1e3 * CHEM_ACC, color="green", ls="--"); ax.axhline(SAFETY * 1e3 * CHEM_ACC, color="green", ls=":")
    ax.set(xscale="log", yscale="log", xlabel="CNOT", ylabel="Erreur (mHa)", title=m); ax.grid(alpha=0.3, which="both")
axes[0][0].legend(fontsize=8)
plt.tight_layout(); plt.show()

## 7. Validation sur des molécules jamais vues

C'est **le test qui compte** : la molécule finale sera différente. On compare la meilleure configuration, la référence et les 3 meilleures configurations suivantes (si la n°1 a sur-appris les molécules d'entraînement, une autre généralisera mieux).

In [ ]:
# On écarte les configs qui produisent exactement les mêmes circuits (même CNOT total et même erreur) :
# la validation teste ainsi 4 comportements différents, pas 4 fois le même circuit.
top = (summ[summ.score < FAIL / len(TRAIN_MOLECULES)].drop_duplicates("label")
       .assign(_sig=lambda d: d.cnot_total.astype(str) + "_" + d.err_max_mHa.round(4).astype(str))
       .drop_duplicates("_sig").head(4).index.tolist())
cands = {"Référence k-ADAPT QEB k=2": BASELINE_CFG}
for rank, t in enumerate(top, 1):
    cands[f"Top {rank} (essai {t})"] = dict(study.trials[t].params)

val_rows, val_runs = [], {}
for m in VALID_MOLECULES:
    for name, cfg in cands.items():
        row, run = run_config(m, cfg, label=name)
        val_rows.append(row); val_runs[(m, name)] = run
        print(f"{m:10s} {name:28s} err={row['erreur_mHa']:8.4f} mHa  CNOT={row['n_cnot']:5d} (simplifié {row['n_cnot_simplifie']:5d})  "
              f"circuits={row['n_circuits_hw']:7d}  valide={row['valide']}  arrêt={row['arret']} "
              f"({row['n_escapes']} sorties de vallée)  {row['temps_s']:.0f}s")
val_df = pd.DataFrame(val_rows)
p = os.path.join(RESULTS_DIR, "validation.csv")
val_df.to_csv(p, mode="a", header=not os.path.exists(p), index=False)

fig, axes = plt.subplots(len(VALID_MOLECULES), 3, figsize=(18, 3.2 * len(VALID_MOLECULES)), squeeze=False)
for i, m in enumerate(VALID_MOLECULES):
    d = val_df[val_df.molecule == m].set_index("label")
    for j, (col, lab, log) in enumerate([("erreur_mHa", "Erreur (mHa)", True), ("n_cnot", "CNOT", False),
                                         ("n_circuits_hw", "Circuits QPU", True)]):
        ax = axes[i][j]
        v = np.maximum(d[col].astype(float), 1e-6) if log else d[col]
        col_ = ["#2ca02c" if ok else "#d62728" for ok in d.precision_chimique] if col == "erreur_mHa" else "tab:blue"
        ax.barh(d.index, v, color=col_, edgecolor="k")
        if col == "erreur_mHa": ax.axvline(1e3 * CHEM_ACC, color="green", ls="--")
        if log: ax.set_xscale("log")
        ax.invert_yaxis(); ax.set_title(f"{m} — {lab}"); ax.grid(axis="x", alpha=0.3)
        if j: ax.set_yticklabels([])
plt.tight_layout(); plt.show()

## 8. Gelé **et** non gelé : pipeline complet

L'énergie de référence du jury peut être celle **avec** gel de cœur (FreezeCore) **ou sans**. On calcule donc les deux, en réutilisant le premier pour accélérer le second :

1. **k-ADAPT dans l'espace gelé** (petit, rapide) → $E_0^{gel}$ ;
2. on **transpose l'ansatz trouvé** dans l'espace complet (décalage des qubits de 2 × nb d'orbitales de cœur, mêmes angles), puis **on continue k-ADAPT** avec le pool complet (qui contient aussi les excitations du cœur) → $E_0^{complet}$.

L'étape 2 démarre déjà presque convergée (il ne reste que la corrélation du cœur, ~qq dixièmes de mHa), donc elle coûte beaucoup moins qu'un k-ADAPT complet partant de Hartree-Fock. Le temps affiché **inclut** la construction des hamiltoniens.

In [ ]:
def shift_op(op, s):
    kind, w = op
    if kind == "FD": return (kind, (tuple(i + s for i in w[0]), tuple(i + s for i in w[1])))
    if kind == "P":  return (kind, (w[0], tuple(i + s for i in w[1])))
    return (kind, tuple(i + s for i in w))

def run_pipeline(name, cfg, label="notre pipeline", max_ops=None):
    # Retourne les lignes 'gelé' et 'complet' (+ temps total, constructions des hamiltoniens incluses).
    p_frozen = get_problem(name)
    fn = full_name(name)
    get_problem(fn)                     # construction comptée via t_build (pas deux fois)
    t0 = time.time()
    out = {}
    if fn == name:                      # rien à geler (ex. H3+) : un seul calcul
        row, run = run_config(name, cfg, label=label, max_ops=max_ops)
        out["gelé"] = out["complet"] = (row, run)
    else:
        row_g, run_g = run_config(name, cfg, label=label + " [gelé]", max_ops=max_ops)
        s = 2 * p_frozen["n_core"]
        init = ([shift_op(o, s) for o in run_g["ops"]], run_g["x"])
        row_c, run_c = run_config(fn, dict(cfg, **FULL_STAGE), label=label + " [complet, warm start]", init=init,
                                  max_ops=None if max_ops is None else max_ops + len(run_g["ops"]))
        row_c["ops_ajoutes_complet"] = len(run_c["ops"]) - len(run_g["ops"])
        out["gelé"], out["complet"] = (row_g, run_g), (row_c, run_c)
    t_total = time.time() - t0 + sum(get_problem(n)["t_build"] for n in {name, fn})
    return out, t_total

# Démonstration + erreur due au gel de cœur
pipe_rows = []
for m in COMPARE_MOLECULES:
    out, t_tot = run_pipeline(m, best_cfg)
    rg, rc = out["gelé"][0], out["complet"][0]
    pf = get_problem(full_name(m))
    print(f"{m:8s} | gelé : E={rg['E']:.6f} (err {rg['erreur_mHa']:.4f} mHa, {rg['n_cnot']} CNOT) | "
          f"complet : E={rc['E']:.6f} (err {rc['erreur_mHa']:.4f} mHa, {rc['n_cnot']} CNOT, {rc['n_qubits']} qubits, "
          f"+{rc.get('ops_ajoutes_complet', 0)} op.) | "
          f"erreur du gel = {1e3*(get_problem(m)['E_fci']-pf['E_fci']):.4f} mHa | {t_tot:.1f}s")
    pipe_rows += [dict(rg, espace="gelé"), dict(rc, espace="complet", temps_total_s=t_tot)]

## 9. Comparaison avec un VQE classique : ratio de gain

**VQE classique** = la recette standard : UCCSD fermionique complet, **sans gel de cœur**, départ Hartree-Fock, optimiseur `CLASSIC_OPTIMIZER` (v5 : BFGS + gradients exacts par défaut ; COBYLA sans gradient restait bloqué à Hartree-Fock sur BeH₂). On note aussi le **temps pour atteindre la précision chimique** (`t_precision_chimique_s`, vide si jamais atteinte). **Notre code** = le pipeline du §8 (k-ADAPT gelé + extension non gelée) avec la meilleure configuration.

Les deux donnent l'énergie **non gelée** ; on compare erreur, CNOT et temps d'exécution total (construction du hamiltonien incluse).

In [ ]:
class _TimeBudget(Exception):
    pass

def run_classic(name, optimizer=CLASSIC_OPTIMIZER, maxiter=CLASSIC_MAXITER, time_limit_min=CLASSIC_TIME_LIMIT_MIN):
    prob = get_problem(full_name(name))   # construction comptée via t_build
    t0 = time.time()
    ops = fermionic(prob)
    est = Estimator(prob)
    x0 = np.zeros(len(ops))
    best = dict(E=np.inf, x=x0, last_print=t0, t_chem=np.nan)
    def _check_budget():
        if time.time() - t0 > 60 * time_limit_min:
            raise _TimeBudget
    def f(x):
        _check_budget()
        now = time.time()
        E = est.energy(ops, x)
        if E < best["E"]:
            best.update(E=E, x=np.array(x))
            if np.isnan(best["t_chem"]) and E - prob["E_fci"] < CHEM_ACC:
                best["t_chem"] = now - t0 + prob["t_build"]
        if now - best["last_print"] > 30:      # signe de vie toutes les 30 s
            best["last_print"] = now
            print(f"      {now - t0:5.0f}s | {est.n_energy} évaluations | meilleure erreur = "
                  f"{1e3 * (best['E'] - prob['E_fci']):.3f} mHa", flush=True)
        return E
    print(f"   VQE classique {name} : {len(ops)} paramètres, {prob['nq']} qubits, budget {time_limit_min} min ...")
    converge = True
    try:
        if optimizer == "COBYLA":
            minimize(f, x0, method="COBYLA", tol=1e-9, options=dict(maxiter=maxiter, rhobeg=0.1))
        else:                               # "BFGS" : UCCSD + gradients exacts (VQE classique "optimisé")
            def jac(x):
                _check_budget()
                return est.grad(ops, x)
            minimize(f, x0, jac=jac, method="BFGS", options=dict(gtol=1e-6, maxiter=maxiter))
    except _TimeBudget:
        converge = False                    # on garde le meilleur point trouvé dans le budget
    t_run = time.time() - t0
    stats = est.final_stats(ops, best["x"])
    err = abs(stats["E"] - prob["E_fci"])
    return dict(molecule=name, methode=f"VQE classique (UCCSD fermionique, {optimizer}, sans gel)",
                n_qubits=prob["nq"], E=stats["E"], erreur_mHa=1e3 * err, precision_chimique=err < CHEM_ACC,
                **circuit_resources(ops), n_eval_E=est.n_energy, converge=converge,
                temps_total_s=t_run + prob["t_build"], t_precision_chimique_s=best["t_chem"])

cmp_rows = []
for m in (COMPARE_MOLECULES if RUN_CLASSIC else []):
    _PROBLEMS.pop(full_name(m), None)          # on refait la construction du hamiltonien pour un chrono honnête
    rc = run_classic(m)
    _PROBLEMS.pop(full_name(m), None); _PROBLEMS.pop(m, None)
    out, t_ours = run_pipeline(m, best_cfg)
    ro = out["complet"][0]
    ours = dict(molecule=m, methode="Notre code (k-ADAPT auto, gelé → complet)", n_qubits=ro["n_qubits"],
                E=ro["E"], erreur_mHa=ro["erreur_mHa"], precision_chimique=ro["precision_chimique"],
                n_cnot=ro["n_cnot"], n_portes=ro["n_portes"], profondeur=ro["profondeur"],
                n_params=ro["n_params"], temps_total_s=t_ours,
                E_gele=out["gelé"][0]["E"], erreur_gele_mHa=out["gelé"][0]["erreur_mHa"])
    ours["gain_temps_x"] = rc["temps_total_s"] / t_ours
    ours["gain_cnot_x"] = rc["n_cnot"] / max(ro["n_cnot"], 1)
    ours["classique_converge"] = rc["converge"]   # si False : le classique a été coupé -> gain de temps = borne basse
    ours["arret"] = ro["arret"]
    cmp_rows += [rc, ours]
    ge = "" if rc["converge"] else "≥"
    print(f"{m:8s} | classique : {rc['erreur_mHa']:8.4f} mHa, {rc['n_cnot']:6d} CNOT, {rc['temps_total_s']:7.1f}s"
          f"{'' if rc['converge'] else ' (coupé, non convergé)'} | "
          f"nous : {ro['erreur_mHa']:8.4f} mHa, {ro['n_cnot']:5d} CNOT, {t_ours:6.1f}s | "
          f"gain temps {ge}×{ours['gain_temps_x']:.1f}, gain CNOT ×{ours['gain_cnot_x']:.1f}")
    pd.DataFrame(cmp_rows).to_csv(os.path.join(RESULTS_DIR, "comparaison_classique_en_cours.csv"), index=False)

cmp_df = pd.DataFrame(cmp_rows)
if cmp_df.empty:
    raise SystemExit("Comparaison sautée (RUN_CLASSIC = False) : passez directement au §10.")
p = os.path.join(RESULTS_DIR, "comparaison_classique.csv")
cmp_df.assign(run_tag=RUN_TAG, cfg=cfg_label(best_cfg)).to_csv(p, mode="a", header=not os.path.exists(p), index=False)
display(cmp_df[["molecule", "methode", "n_qubits", "erreur_mHa", "precision_chimique", "converge", "classique_converge",
                "n_cnot", "temps_total_s", "t_precision_chimique_s", "gain_temps_x", "gain_cnot_x"]].round(4))

In [ ]:
mols_c = list(dict.fromkeys(cmp_df.molecule))
cl = cmp_df[cmp_df.methode.str.startswith("VQE")].set_index("molecule").loc[mols_c]
us = cmp_df[cmp_df.methode.str.startswith("Notre")].set_index("molecule").loc[mols_c]
xx = np.arange(len(mols_c)); w = 0.38
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
for ax, col, lab, ratio in [(axes[0], "temps_total_s", "Temps d'exécution total (s)", "gain_temps_x"),
                            (axes[1], "n_cnot", "Nombre de CNOT", "gain_cnot_x"),
                            (axes[2], "erreur_mHa", "Erreur vs FCI non gelée (mHa)", None)]:
    a = np.maximum(cl[col].astype(float), 1e-6); b = np.maximum(us[col].astype(float), 1e-6)
    ax.bar(xx - w / 2, a, w, label="VQE classique", color="#999999", edgecolor="k")
    ax.bar(xx + w / 2, b, w, label="Notre code", color="#1f77b4", edgecolor="k")
    if ratio:
        for i, r in enumerate(us[ratio]):
            ge = "≥" if (ratio == "gain_temps_x" and not bool(us["classique_converge"].iloc[i])) else ""
            ax.text(xx[i], max(a.iloc[i], b.iloc[i]) * 1.15, f"{ge}×{r:.1f}", ha="center", fontweight="bold")
    if col == "erreur_mHa":
        ax.axhline(1e3 * CHEM_ACC, color="green", ls="--", label="précision chimique")
    ax.set_yscale("log"); ax.set_xticks(xx, mols_c); ax.set_title(lab); ax.grid(axis="y", alpha=0.3)
axes[0].legend()
plt.suptitle(f"Gain de notre code vs VQE classique — config : {cfg_label(best_cfg)}", fontsize=10)
plt.tight_layout(); plt.show()

## 10. Molécule cible

Ajoutez la molécule dans `MOLECULES` (avec son espace actif gelé), mettez son nom dans `TARGET_MOLECULE` et choisissez la configuration qui a **le mieux validé** au §7 (pas forcément le Top 1). On calcule $E_0$ **gelé et non gelé** ; les deux circuits sont sauvegardés en JSON.

In [ ]:
FINAL_CFG = best_cfg            # ou cands["Top 2 (essai ..)"], ou un dict écrit à la main

if TARGET_MOLECULE:
    mo = MAX_OPS
    while True:
        out, t_tot = run_pipeline(TARGET_MOLECULE, FINAL_CFG, label="final", max_ops=mo)
        if not any(r["arret"] == "max_ops" for r, _ in out.values()) or mo >= 4 * MAX_OPS:
            break
        mo *= 2                      # limite atteinte : on relance avec une limite doublée (au plus 4 x MAX_OPS)
        print(f"⚠️  limite d'opérateurs atteinte -> relance avec max_ops = {mo}")
    for espace, (row, run) in out.items():
        print(f"[{espace:7s}] E0 = {row['E']:.8f} Ha | E_FCI = {row['E_fci']:.8f} Ha | erreur = {row['erreur_mHa']:.4f} mHa"
              f" | {row['n_qubits']} qubits, {row['n_cnot']} CNOT ({row['n_cnot_simplifie']} après simplification), {row['n_portes']} portes, prof. {row['profondeur']},"
              f" {row['n_params']} params, valide = {row['valide']}, arrêt = {row['arret']}")
        with open(os.path.join(RESULTS_DIR, f"circuit_{TARGET_MOLECULE}_{espace}.json"), "w") as f:
            json.dump(dict(cfg=FINAL_CFG, E=row["E"], ops=[[k, str(w)] for k, w in run["ops"]],
                           params=list(map(float, run["x"]))), f, indent=1)
    print(f"Temps total (gelé + complet, hamiltoniens inclus) : {t_tot:.1f}s")
    for espace, (row, run) in out.items():
        h = run["hist"]
        plt.plot([e["n_ops"] for e in h], np.maximum([e["err_mHa"] for e in h], 1e-6), "o-", label=espace)
    plt.axhline(1e3 * CHEM_ACC, color="green", ls="--"); plt.yscale("log"); plt.legend()
    plt.xlabel("Opérateurs"); plt.ylabel("Erreur (mHa)"); plt.title(TARGET_MOLECULE); plt.grid(alpha=0.3); plt.show()

## 11. Pour itérer

- **Ce que je regarde en priorité** : le tableau du §5 (quel pool est le meilleur à réglages égaux), le Top 10 du §6.1, les boîtes à moustaches du §6.2 (quels hyperparamètres comptent vraiment) et surtout le §7 (est-ce que ça **généralise**).
- **Resserrer l'espace de recherche** : enlevez de `SEARCH_SPACE` les valeurs qui échouent toujours (0 % ok) et ajoutez des valeurs autour des meilleures, puis relancez avec un nouveau `RUN_TAG`.
- **Mettre une molécule de validation dans l'entraînement** si elle échoue (ex. LiH étiré) : la configuration devient plus robuste, au prix d'un peu plus de CNOT.
- Si une ligne affiche `arrêt = max_ops` (ou `⚠MAX` pendant l'auto-tuner), le calcul a été **coupé** par `MAX_OPS` : l'erreur n'est pas fiable, augmentez `MAX_OPS`.
- Les fichiers `results/autotune_trials.csv`, `results/validation.csv` et `results/best_config.json` contiennent tout ce qu'il faut pour comparer les runs.